# ARGUS Phase 5 - notebook 02: rejection-sampling fine-tune (RFT)
Part A samples 8 answers per distinct training input from the SFT adapter. Then YOU run `python -m ml.train_m1.rft_filter` on your laptop (the gate needs OPA), upload `rft_train.jsonl`, and run Part B.

In [ ]:
%%capture
# This is the install cell of Unsloth's own Qwen3.5-4B notebook (copied Oct 2026). If Unsloth has changed it, replace this cell
# by the first code cell of the current notebook at https://github.com/unslothai/notebooks (nb/Qwen3_5_(4B)_Vision.ipynb).
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"


In [ ]:
# ---------- settings you may change ----------
from google.colab import drive
drive.mount("/content/drive")
DATA_DIR  = "/content/drive/MyDrive/argus/data"      # the folder where you uploaded ml/train_m1/data/*
OUT_DIR   = "/content/drive/MyDrive/argus/out"       # adapters and predictions are saved here
BASE_MODEL = "unsloth/Qwen3.5-4B"
import os, json, random, hashlib
os.makedirs(OUT_DIR, exist_ok=True)

def read_jsonl(path):
    with open(path, encoding="utf-8") as h:
        return [json.loads(l) for l in h if l.strip()]
SFT_ADAPTER = ""    # path of the chosen SFT checkpoint folder in Drive
MAX_LEN = 1024


In [ ]:
import torch
from unsloth import FastLanguageModel

def load_model(name, max_len):
    try:
        m, t = FastLanguageModel.from_pretrained(name, max_seq_length=max_len, load_in_4bit=False, use_gradient_checkpointing="unsloth")
    except Exception as e:   # Qwen3.5 is a vision-language model; fall back to the loader Unsloth's own notebook uses
        print("FastLanguageModel failed (%s); using FastVisionModel" % str(e)[:150])
        from unsloth import FastVisionModel
        m, t = FastVisionModel.from_pretrained(name, load_in_4bit=False, use_gradient_checkpointing="unsloth")
    return m, t

def add_lora(m):
    return FastLanguageModel.get_peft_model(m, r=16, lora_alpha=16, lora_dropout=0, bias="none", random_state=3407,
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"], use_gradient_checkpointing="unsloth")

def text_tok(t):
    return getattr(t, "tokenizer", t)   # a processor wraps the real tokenizer

def prompt_text(tok, row):
    msgs = [{"role":"system","content":row["system"]},{"role":"user","content":row["user"]}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)

def encode(tok, row, max_len):
    p = tok(prompt_text(tok, row), add_special_tokens=False)["input_ids"]
    c = tok(row["assistant"] + "<|im_end|>", add_special_tokens=False)["input_ids"]
    ids = (p + c)[:max_len]
    labels = ([-100] * len(p) + c)[:max_len]
    return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": labels}

@torch.no_grad()
def generate_greedy(model, tok, rows, max_new=300, batch=16, sample=None):
    tok.padding_side = "left"
    outs = []
    for i in range(0, len(rows), batch):
        chunk = rows[i:i+batch]
        enc = tok([prompt_text(tok, r) for r in chunk], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        kw = dict(max_new_tokens=max_new, use_cache=True)
        if sample: kw.update(do_sample=True, temperature=sample["temperature"], top_p=0.95, num_return_sequences=sample["n"])
        else: kw.update(do_sample=False)
        gen = model.generate(**enc, **kw)
        new = gen[:, enc["input_ids"].shape[1]:]
        texts = tok.batch_decode(new, skip_special_tokens=True)
        n = sample["n"] if sample else 1
        for j in range(len(chunk)):
            outs.append([t.strip() for t in texts[j*n:(j+1)*n]])
    return outs


## Part A - sample candidates

In [ ]:
assert SFT_ADAPTER, "set SFT_ADAPTER first"
rows = read_jsonl(f"{DATA_DIR}/sft_train.jsonl")
seen, distinct = set(), []
for r in rows:
    if r["input_key"] not in seen:
        seen.add(r["input_key"]); distinct.append(r)
print(len(distinct), "distinct training inputs")
m, t = load_model(SFT_ADAPTER, MAX_LEN)
FastLanguageModel.for_inference(m)
tk = text_tok(t)
outs = generate_greedy(m, tk, distinct, batch=4, sample={"n": 8, "temperature": 0.8})
with open(f"{OUT_DIR}/rft_candidates.jsonl", "w", encoding="utf-8") as h:
    for r, o in zip(distinct, outs): h.write(json.dumps({"input_key": r["input_key"], "contents": o}) + "\n")
print("written", f"{OUT_DIR}/rft_candidates.jsonl", "-> download it and run rft_filter on your laptop")


## Part B - train on the filtered set (run after uploading rft_train.jsonl to the data folder)

In [ ]:
from datasets import Dataset
from transformers import DataCollatorForSeq2Seq
from trl import SFTTrainer, SFTConfig
train_rows = read_jsonl(f"{DATA_DIR}/rft_train.jsonl")
held = set(open(f"{DATA_DIR}/heldout_hashes.txt").read().split())
assert not [r for r in train_rows if r["full_hash"] in held], "LEAKAGE"
model, tokenizer = load_model(SFT_ADAPTER, MAX_LEN)      # continues from the SFT adapter
tok = text_tok(tokenizer)
model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=16, lora_dropout=0, bias="none", random_state=3407,
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"], use_gradient_checkpointing="unsloth") if not hasattr(model, "peft_config") else model
ds = Dataset.from_list([encode(tok, r, MAX_LEN) for r in train_rows])
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds,
    data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100),
    args=SFTConfig(per_device_train_batch_size=8, gradient_accumulation_steps=2, warmup_steps=5, num_train_epochs=1, learning_rate=1e-4,
        logging_steps=5, optim="adamw_8bit", weight_decay=0.001, lr_scheduler_type="linear", seed=3407, output_dir=f"{OUT_DIR}/rft",
        report_to="none", save_strategy="epoch", remove_unused_columns=False, dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True}, max_length=MAX_LEN))
trainer.train()
model.save_pretrained(f"{OUT_DIR}/rft_adapter"); tok.save_pretrained(f"{OUT_DIR}/rft_adapter")
print("saved", f"{OUT_DIR}/rft_adapter")


In [ ]:
# Export to GGUF
m, t = load_model(f"{OUT_DIR}/rft_adapter", 2048)
m.save_pretrained_gguf(f"{OUT_DIR}/argus_rft_gguf", t, quantization_method="q4_k_m")
print(os.listdir(f"{OUT_DIR}/argus_rft_gguf"))
